# 🐾 Cat's Eye — AI Video Clipper (Google Colab Edition)

**Turn long videos and live streams into captioned 9:16 vertical shorts with AI.**

### ⚡ Google Drive Source of Truth
- All durable checkpoints, database state, rendered clips, and model caches live in `Google Drive / MyDrive / CatsEye`.
- If your Colab session times out or disconnects unexpectedly, **simply reconnect and run this notebook again**.
- Cat's Eye will automatically restore the latest verified database checkpoint and continue running any interrupted jobs.

In [ ]:
# Step 1: Mount Google Drive (Required for Durable Storage)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 2: Clone Cat's Eye & Install Dependencies
import os

APP_DIR = '/content/cats-eye'
REPO_URL = 'https://github.com/wajii-knows/cats-eye.git'

if not os.path.exists(f'{APP_DIR}/.git'):
    !git clone {REPO_URL} {APP_DIR}
else:
    !git -C {APP_DIR} pull --ff-only

%cd /content/cats-eye

# Ensure ffmpeg is installed and install Python dependencies
!apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null
!pip install -q -r requirements.txt
print('✅ Cat\'s Eye is installed and ready.')

In [ ]:
# Step 3: Launch Cat's Eye & Open Web UI
import os
from google.colab.output import eval_js

# Load optional Colab Secrets (PEAKCUT_OWNER_PASSWORD, PEAKCUT_AI_KEY)
try:
    from google.colab import userdata
    for k in ('PEAKCUT_OWNER_PASSWORD', 'PEAKCUT_AI_KEY'):
        try:
            v = userdata.get(k)
            if v:
                os.environ[k] = v.strip()
        except Exception:
            pass
except Exception:
    pass

os.environ.setdefault("PEAKCUT_OWNER_PASSWORD", "cats-eye-colab-2026")
os.environ["PEAKCUT_COLAB"] = "1"

# Official Google Colab Native Port Proxy Link
proxy_url = eval_js("google.colab.kernel.proxyPort(8765)")
print("=" * 60)
print("🐾 CLICK TO OPEN CAT'S EYE IN YOUR BROWSER (same signed-in Chrome/Edge profile):")
print(proxy_url)
print(f"🔑 Password: {os.environ['PEAKCUT_OWNER_PASSWORD']}")
print("=" * 60)

%cd /content/cats-eye
# Start Cat's Eye with automated lease acquisition and checkpoint recovery
!python scripts/colab_bootstrap.py

### ℹ️ Disconnection & Recovery FAQ
- **What happens if Colab disconnects?**
  Nothing is lost. The database is snapshotted every 30 seconds to Google Drive, and intra-stage chunks (transcripts, clips) are uploaded to Drive upon completion.
- **Can I run multiple notebooks at once?**
  Cat's Eye enforces a session lease file (`lease.json`) on your Google Drive. If an old session died, the new session will automatically take over after 3 minutes, or you can supply `--force-lease`.
- **Terms of Service Compliance:**
  We use Google Colab's native `google.colab.kernel.proxyPort(8765)`. It runs directly within Google's authenticated infrastructure without external public reverse tunnels (like cloudflared or ngrok), remaining 100% compliant with Colab free-tier rules.